<a href="https://colab.research.google.com/github/daniyal1d/Document_QA_Bot_for_Colab/blob/main/Document_QA_Bot_for_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.2 MB/s eta 0:00:00


In [ ]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.1 MB/s eta 0:00:00
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [ ]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00


In [ ]:
import os
from PyPDF2 import PdfReader
from langchain.chains import RetrievalQA
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
import google.generativeai as genai
from urllib.request import urlopen
from google.colab import files  # Import files for file upload

def load_document(file_path):
    """
    Loads text from a PDF or TXT file.  Handles URLs and local files.

    Args:
        file_path (str): Path to the PDF or TXT file, or a URL.

    Returns:
        str: The extracted text from the document, or an empty string on error.
    """
    text = ""
    try:
        if file_path.lower().startswith("http://") or file_path.lower().startswith("https://"):
            # Handle URL
            try:
                with urlopen(file_path) as response:
                    if file_path.lower().endswith(".pdf"):
                        reader = PdfReader(response)
                        for page in reader.pages:
                            text += page.extract_text() or ""
                    elif file_path.lower().endswith(".txt"):
                        text = response.read().decode('utf-8')
                    else:
                        print("Unsupported file type from URL. Please provide a .pdf or .txt file.")
                        return ""
            except Exception as e:
                print(f"Error loading document from URL: {e}")
                return ""

        elif file_path == "local":  # Handle local file upload from Colab
             uploaded = files.upload() #this returns a dictionary of the files uploaded
             for filename in uploaded.keys(): #iterate over the filenames.
                if filename.lower().endswith(".pdf"):
                    # Handle local PDF
                    try:
                        # with open(filename, "rb") as file: # No need to open, already in memory
                        reader = PdfReader(uploaded[filename]) # Pass the file object from uploaded
                        for page in reader.pages:
                            text += page.extract_text() or ""
                    except Exception as e:
                        print(f"Error loading local PDF: {e}")
                        return ""
                elif filename.lower().endswith(".txt"):
                    # Handle local TXT
                    try:
                        # with open(filename, "r", encoding="utf-8") as file:
                        text = uploaded[filename].decode('utf-8')
                    except Exception as e:
                        print(f"Error loading local TXT: {e}")
                        return ""
                else:
                    print("Unsupported file type. Please provide a .pdf or .txt file.")
                    return ""
        elif file_path.lower().endswith(".pdf"):
            # Handle local PDF
            try:
                with open(file_path, "rb") as file:
                    reader = PdfReader(file)
                    for page in reader.pages:
                        text += page.extract_text() or ""
            except Exception as e:
                print(f"Error loading local PDF: {e}")
                return ""
        elif file_path.lower().endswith(".txt"):
            # Handle local TXT
            try:
                with open(file_path, "r", encoding="utf-8") as file:
                    text = file.read()
            except Exception as e:
                print(f"Error loading local TXT: {e}")
                return ""
        else:
            print("Unsupported file type. Please provide a .pdf or .txt file.")
            return ""
    except Exception as e:
        print(f"Error loading document: {e}")
        return ""
    return text


def split_text(text, chunk_size=1000, chunk_overlap=200):
    """
    Splits the text into chunks using RecursiveCharacterTextSplitter.

    Args:
        text (str): The text to split.
        chunk_size (int, optional): The maximum size of each chunk. Defaults to 1000.
        chunk_overlap (int, optional): The overlap between adjacent chunks. Defaults to 200.

    Returns:
        list: A list of text chunks.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    chunks = text_splitter.split_text(text)
    return chunks


def create_vector_store(chunks, embedding_model="models/embedding-001"):
    """
    Creates a FAISS vector store from the text chunks using the specified embedding model.

    Args:
        chunks (list): A list of text chunks.
        embedding_model (str, optional): The name of the Gemini embedding model to use.
            Defaults to "models/embedding-001".

    Returns:
        FAISS: The FAISS vector store.
    """
    embeddings = GoogleGenerativeAIEmbeddings(model=embedding_model)
    vector_store = FAISS.from_texts(chunks, embeddings)
    return vector_store


def create_qa_chain(vector_store, llm_model="models/chat-bison-001", temperature=0.3):
    """
    Creates a RetrievalQA chain using the specified vector store and language model.

    Args:
        vector_store (FAISS): The FAISS vector store.
        llm_model (str, optional): The name of the Gemini language model to use.
            Defaults to "models/chat-bison-001".
        temperature (float, optional): The temperature for the language model.

    Returns:
        RetrievalQA: The RetrievalQA chain.
    """
    llm = ChatGoogleGenerativeAI(model_name=llm_model, temperature=temperature)
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm, chain_type="stuff", retriever=vector_store.as_retriever()
    )
    return qa_chain


def ask_question(qa_chain, question):
    """
    Asks a question to the QA chain and prints the answer.

    Args:
        qa_chain (RetrievalQA): The RetrievalQA chain.
        question (str): The question to ask.
    """
    try:
        answer = qa_chain.run(question)
        print(f"Question: {question}")
        print(f"Answer: {answer}")
    except Exception as e:
        print(f"Error asking question: {e}")
        print("Please check your query and ensure the document is loaded correctly.")



def main(file_path, question):
    """
    Main function to run the Document QA Bot.

    Args:
        file_path (str): Path to the PDF or TXT file.
        question (str): The question to ask about the document.
    """
    text = load_document(file_path)
    if not text:
        print("Failed to load document. Exiting.")
        return

    chunks = split_text(text)
    vector_store = create_vector_store(chunks)
    qa_chain = create_qa_chain(vector_store)
    ask_question(qa_chain, question)



if __name__ == "__main__":
    # Get file path and question from the user
    # file_path = input("Enter the path to your PDF or TXT file: ")
    # question = input("Enter your question: ")

    # Use a sample PDF URL for Colab
    # pdf_url = "https://www.cs.princeton.edu/sites/default/files/uploads/kariger_on_being_a_senior.pdf"  # Replace with a direct link to a PDF
    # question = "What is the main topic of the document?"

    # Use "local" to trigger file upload in Colab
    file_path = "local"
    question = "What is the main topic of the document?"

    # Call the main function
    main(file_path, question)


Saving AI_in_Education_Sample.pdf to AI_in_Education_Sample.pdf
Error loading local PDF: 'bytes' object has no attribute 'seek'
Failed to load document. Exiting.
